In [10]:
import os
from pinecone import Pinecone
if not os.getenv("PINECONE_API_KEY"):
    os.environ["PINECONE_API_KEY"] = os.getenv("PINECONE_API_KEY")

pinecone_api_key = os.environ.get("PINECONE_API_KEY")
pc= Pinecone(api_key=pinecone_api_key)
pc

In [11]:
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
load_dotenv()

# Keep the embedding dimension aligned with the Pinecone index.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1024)
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x7fb6d46dae90>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x7fb6d46db750>, model='text-embedding-3-small', dimensions=1024, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base='https://models.inference.ai.azure.com', openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [12]:
## connecting to db
from pinecone import ServerlessSpec

index_name = "rag"  # change if desired

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1024,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)
index

In [13]:
# creating pinecone vector store
from langchain_pinecone import PineconeVectorStore
vectorstore = PineconeVectorStore(
    index= index,
    embedding=embeddings
)
vectorstore

In [14]:
# sample documents to be added
from langchain_core.documents import Document

sample_documents=[
    Document(
        page_content=""" The Great Wall of China is a series of fortifications made of stone, brick, tamped earth, wood, and other materials. It was built along the northern borders of China to protect against invasions and raids by nomadic groups. The wall stretches over 13,000 miles and is considered one of the greatest architectural feats in history.""",
        metadata={"source": "Wikipedia", "topic": "Great Wall of China", "page": 1}
    ),
    Document(
        page_content =""" The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars in Paris, France. It was named after the engineer Gustave Eiffel, whose company designed and built the tower. The Eiffel Tower is one of the most recognizable structures in the world and is a global cultural icon of France.""",
        metadata={"source": "Wikipedia", "topic": "Eiffel Tower", "page": 1}
    ),
    Document(
        page_content =""" The Pyramids of Giza are ancient pyramid-shaped masonry structures located in Giza, Egypt. They were built as tombs for the pharaohs and their consorts during the Old Kingdom period. The Great Pyramid of Giza is the largest and most famous of the three pyramids and is one of the Seven Wonders of the Ancient World.""",
        metadata={"source": "Wikipedia", "topic": "Pyramids of Giza", "page": 1}
    ),
    Document(
        page_content =""" The Taj Mahal is an ivory-white marble mausoleum located in Agra, India. It was commissioned by the Mughal emperor Shah Jahan in memory of his favorite wife, Mumtaz Mahal. The Taj Mahal is widely recognized as a symbol of love and is one of the most famous buildings in the world.""",
        metadata={"source": "Wikipedia", "topic": "Taj Mahal", "page": 1}   
    ),
    Document(
        page_content =""" The Statue of Liberty is a colossal neoclassical sculpture on Liberty Island in New York Harbor, USA. It was a gift from the people of France to the United States and was designed by French sculptor Frédéric Auguste Bartholdi. The statue is a symbol of freedom and democracy and is one of the most iconic landmarks in the world.""",
        metadata={"source": "Wikipedia", "topic": "Statue of Liberty", "page": 1}
    )
]

In [15]:
# adding documents to the pinecone index
vectorstore.add_documents(sample_documents)

['916c24d0-d8c6-4c09-aa94-6bfb521a240a',
 '540b44a4-0c99-49e5-8703-774632321947',
 '028b1cb5-40b7-4bed-8e90-9158ca3385ce',
 '026cdffd-b76a-42e5-820e-6b2f228b59cf',
 '1c3e87c3-6e0e-4346-b9c5-ab9d4a561cf2']

In [17]:
# searching for documents 
result1 =vectorstore.similarity_search("Tell me about the Great Wall of China", k=2)
print(result1[0].page_content)
print(result1[1].page_content)

 The Great Wall of China is a series of fortifications made of stone, brick, tamped earth, wood, and other materials. It was built along the northern borders of China to protect against invasions and raids by nomadic groups. The wall stretches over 13,000 miles and is considered one of the greatest architectural feats in history.
 The Pyramids of Giza are ancient pyramid-shaped masonry structures located in Giza, Egypt. They were built as tombs for the pharaohs and their consorts during the Old Kingdom period. The Great Pyramid of Giza is the largest and most famous of the three pyramids and is one of the Seven Wonders of the Ancient World.


In [18]:
# searching with scores
result2 = vectorstore.similarity_search_with_score("Tell me about the Great Wall of China", k=2)
for doc, score in result2:
    print(f"Score: {score}, Content: {doc.page_content}")
    

Score: 0.699644923, Content:  The Great Wall of China is a series of fortifications made of stone, brick, tamped earth, wood, and other materials. It was built along the northern borders of China to protect against invasions and raids by nomadic groups. The wall stretches over 13,000 miles and is considered one of the greatest architectural feats in history.
Score: 0.286421478, Content:  The Pyramids of Giza are ancient pyramid-shaped masonry structures located in Giza, Egypt. They were built as tombs for the pharaohs and their consorts during the Old Kingdom period. The Great Pyramid of Giza is the largest and most famous of the three pyramids and is one of the Seven Wonders of the Ancient World.


In [20]:
# converting to a retriever
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 2})
query = "Tell me about the Great Wall of China"
retrieved_docs = retriever.invoke(query)
for doc in retrieved_docs:
    print(f"Content: {doc.page_content}, Metadata: {doc.metadata}")


Content:  The Great Wall of China is a series of fortifications made of stone, brick, tamped earth, wood, and other materials. It was built along the northern borders of China to protect against invasions and raids by nomadic groups. The wall stretches over 13,000 miles and is considered one of the greatest architectural feats in history., Metadata: {'page': 1.0, 'source': 'Wikipedia', 'topic': 'Great Wall of China'}
Content:  The Pyramids of Giza are ancient pyramid-shaped masonry structures located in Giza, Egypt. They were built as tombs for the pharaohs and their consorts during the Old Kingdom period. The Great Pyramid of Giza is the largest and most famous of the three pyramids and is one of the Seven Wonders of the Ancient World., Metadata: {'page': 1.0, 'source': 'Wikipedia', 'topic': 'Pyramids of Giza'}
